# Results Analysis

Generates all paper-ready figures and tables from trained model results.

Sections:
1. Results matrix (all models × all splits)
2. Transfer gap bar chart
3. Ablation tables (handoff blocks, patch size, augmentation)
4. Few-shot learning curve
5. Per-class error analysis on clinical splits
6. McNemar significance table

In [1]:
import sys; sys.path.insert(0, '..')
import json, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

MODELS  = ['cnn', 'resnet1d', 'transformer', 'hybrid', 'hybrid_ft']
SPLITS  = ['test', '2018clinical', '2019clinical']
EXP_DIR = '../experiments'
COLORS  = {'cnn':'#1D9E75', 'resnet1d':'#534AB7', 'transformer':'#D85A30',
            'hybrid':'#BA7517', 'hybrid_ft':'#185FA5'}

def load_results(exp_name):
    path = os.path.join(EXP_DIR, exp_name, 'results.json')
    if not os.path.exists(path):
        print(f'  [missing] {path}')
        return None
    with open(path) as f:
        return json.load(f)

# Load all results — update experiment names after training
EXP_NAMES = {
    'cnn':         'cnn_best',
    'resnet1d':    'resnet1d_best',
    'transformer': 'transformer_best',
    'hybrid':      'hybrid_best',
    'hybrid_ft':   'hybrid_finetuned',
}
results = {name: load_results(exp) for name, exp in EXP_NAMES.items()}
available = {k: v for k, v in results.items() if v is not None}
print(f'Loaded {len(available)}/{len(results)} model results')

  [missing] ../experiments\cnn_best\results.json
  [missing] ../experiments\resnet1d_best\results.json
  [missing] ../experiments\transformer_best\results.json
  [missing] ../experiments\hybrid_best\results.json
  [missing] ../experiments\hybrid_finetuned\results.json
Loaded 0/5 model results


---
## 1. Results Matrix — all models × all splits

In [ ]:
print('\nResults matrix — Accuracy (Macro-F1)')
print('='*72)
header = f'{"Model":<16}' + ''.join(f'  {s:<18}' for s in SPLITS)
print(header)
print('─'*72)

for model_name, res in available.items():
    row = f'{model_name:<16}'
    for split in SPLITS:
        split_data = res.get('splits', {}).get(split, {})
        m = split_data.get('metrics', {})
        acc = m.get('accuracy', float('nan'))
        f1  = m.get('f1_macro', float('nan'))
        row += f'  {acc:.4f} ({f1:.4f})  '
    print(row)
print('─'*72)
print('Format: accuracy (macro-F1)')

In [ ]:
# Heatmap of accuracy across models and splits
model_names = list(available.keys())
acc_matrix = np.full((len(model_names), len(SPLITS)), np.nan)

for i, mname in enumerate(model_names):
    for j, split in enumerate(SPLITS):
        m = available[mname].get('splits',{}).get(split,{}).get('metrics',{})
        acc_matrix[i, j] = m.get('accuracy', np.nan)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(acc_matrix, cmap='RdYlGn', vmin=0.5, vmax=1.0, aspect='auto')
plt.colorbar(im, ax=ax, label='Accuracy')

ax.set_xticks(range(len(SPLITS)));  ax.set_xticklabels(SPLITS, rotation=15)
ax.set_yticks(range(len(model_names))); ax.set_yticklabels(model_names)

for i in range(len(model_names)):
    for j in range(len(SPLITS)):
        val = acc_matrix[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=10, color='black' if val < 0.85 else 'white')

ax.set_title('Accuracy heatmap — all models × all evaluation splits', fontweight='bold')
plt.tight_layout()
plt.savefig('../experiments/fig_results_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Transfer Gap — source → clinical

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
ood_splits = ['2018clinical', '2019clinical']

for ax, ood_split in zip(axes, ood_splits):
    gaps, labels, colors = [], [], []
    for mname, res in available.items():
        test_acc = res.get('splits',{}).get('test',{}).get('metrics',{}).get('accuracy', np.nan)
        ood_acc  = res.get('splits',{}).get(ood_split,{}).get('metrics',{}).get('accuracy', np.nan)
        if not (np.isnan(test_acc) or np.isnan(ood_acc)):
            gaps.append(test_acc - ood_acc)
            labels.append(mname)
            colors.append(COLORS.get(mname, '#888780'))

    bars = ax.barh(labels, gaps, color=colors, height=0.5)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('Transfer gap (test acc − OOD acc)')
    ax.set_title(f'{ood_split}', fontweight='bold')
    ax.bar_label(bars, fmt='{:.3f}', padding=3, fontsize=9)

axes[0].set_ylabel('Model')
plt.suptitle('Transfer gap: smaller = better generalisation to clinical data', fontsize=12)
plt.tight_layout()
plt.savefig('../experiments/fig_transfer_gap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Ablation — Hybrid handoff blocks

In [ ]:
ablation_dir = '../experiments/ablations'

handoff_results = {}
for n_blocks in [1, 2, 3]:
    path = os.path.join(ablation_dir, f'hybrid_handoff_{n_blocks}', 'results.json')
    if os.path.exists(path):
        with open(path) as f:
            handoff_results[n_blocks] = json.load(f)

if handoff_results:
    fig, axes = plt.subplots(1, len(SPLITS), figsize=(13, 4), sharey=False)
    for ax, split in zip(axes, SPLITS):
        xs = []
        accs = []
        for n_blocks, res in handoff_results.items():
            acc = res.get('splits',{}).get(split,{}).get('metrics',{}).get('accuracy', np.nan)
            xs.append(n_blocks)
            accs.append(acc)
        ax.plot(xs, accs, 'o-', color='#BA7517', lw=2, ms=8)
        ax.set_xticks([1,2,3])
        ax.set_xlabel('CNN blocks before Transformer')
        ax.set_ylabel('Accuracy') if ax == axes[0] else None
        ax.set_title(split, fontweight='bold')
        ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
    plt.suptitle('Ablation: hybrid handoff point (CNN blocks before Transformer)', fontsize=12)
    plt.tight_layout()
    plt.savefig('../experiments/fig_ablation_handoff.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Run: python scripts/run_ablation.py --ablation hybrid_handoff')

---
## 4. Few-shot Learning Curve

In [ ]:
fs_path = os.path.join(ablation_dir, 'few_shot_curve.json')

if os.path.exists(fs_path):
    with open(fs_path) as f:
        fs = json.load(f)

    shots = sorted([int(k) for k in fs.keys()])
    ood_splits_fs = ['2018clinical', '2019clinical']

    fig, ax = plt.subplots(figsize=(8, 5))

    for ood, color, marker in zip(ood_splits_fs, ['#D85A30','#534AB7'], ['o','s']):
        accs = []
        for n in shots:
            ood_res = fs.get(str(n), {}).get('ood', {}).get(ood, {})
            accs.append(ood_res.get('accuracy', np.nan))
        ax.plot(shots, accs, f'{marker}-', color=color, lw=2, ms=8, label=ood)

    # Baseline: zero-shot (from main results)
    for ood, color in zip(ood_splits_fs, ['#D85A30','#534AB7']):
        if 'hybrid' in available:
            baseline = available['hybrid'].get('splits',{}).get(ood,{}).get('metrics',{}).get('accuracy', np.nan)
            ax.axhline(baseline, color=color, lw=1, linestyle='--', alpha=0.5)

    ax.set_xscale('log')
    ax.set_xticks(shots); ax.set_xticklabels(shots)
    ax.set_xlabel('Fine-tuning samples per class')
    ax.set_ylabel('Clinical accuracy')
    ax.legend(title='OOD split')
    ax.set_title('Few-shot adaptation learning curve\n(dashed = zero-shot baseline)', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../experiments/fig_few_shot_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Run: python scripts/run_ablation.py --ablation few_shot --pretrained experiments/hybrid_best')

---
## 5. Per-class Error Analysis (Clinical Splits)

In [ ]:
SHARED_CLASSES = [0, 2, 3, 5, 6]

for ood_split in ['2018clinical', '2019clinical']:
    fig, axes = plt.subplots(1, len(available), figsize=(4*len(available), 4), sharey=True)
    if len(available) == 1: axes = [axes]

    for ax, (mname, res) in zip(axes, available.items()):
        split_data = res.get('splits', {}).get(ood_split, {})
        metrics = split_data.get('metrics', {})
        f1s = [metrics.get(f'f1_class_{c}', np.nan) for c in SHARED_CLASSES]
        colors = ['#1D9E75' if f >= 0.8 else '#D85A30' if f < 0.6 else '#BA7517'
                  for f in f1s]
        bars = ax.bar([str(c) for c in SHARED_CLASSES], f1s, color=colors)
        ax.set_ylim(0, 1.05)
        ax.axhline(0.8, color='gray', lw=0.8, linestyle='--')
        ax.set_title(mname, fontweight='bold')
        ax.set_xlabel('Class')
        ax.bar_label(bars, fmt='{:.2f}', padding=2, fontsize=8)

    axes[0].set_ylabel('Per-class F1')
    plt.suptitle(f'Per-class F1 on {ood_split} (green ≥ 0.8, red < 0.6)', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'../experiments/fig_per_class_{ood_split}.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 6. McNemar Pairwise Significance

In [ ]:
import math

def chi2_sf(x):
    return math.erfc(math.sqrt(x/2)) if x > 0 else 1.0

def mcnemar(pA, pB, targets):
    pA, pB, t = np.array(pA), np.array(pB), np.array(targets)
    cA, cB = pA==t, pB==t
    n_ab = ((cA)&(~cB)).sum(); n_ba = ((~cA)&(cB)).sum()
    n = n_ab + n_ba
    if n == 0: return {'p': 1.0, 'sig': False}
    stat = (abs(n_ab-n_ba)-1)**2 / n
    p = chi2_sf(stat)
    return {'stat': float(stat), 'p': float(p), 'sig': p < 0.05, 'A_wins': int(n_ab), 'B_wins': int(n_ba)}

model_list = list(available.keys())
print('McNemar pairwise significance tests (test split)')
print(f'{"Pair":<32} {"stat":>8} {"p-value":>10} {"Sig?":>6}')
print('─'*58)

for i in range(len(model_list)):
    for j in range(i+1, len(model_list)):
        mA, mB = model_list[i], model_list[j]
        pA = available[mA].get('splits',{}).get('test',{}).get('predictions',[])
        pB = available[mB].get('splits',{}).get('test',{}).get('predictions',[])
        tg = available[mA].get('splits',{}).get('test',{}).get('targets',[])
        if pA and pB and tg:
            r = mcnemar(pA, pB, tg)
            sig = '***' if r['sig'] else 'n.s.'
            print(f'{mA} vs {mB:<20} {r["stat"]:>8.3f} {r["p"]:>10.4f} {sig:>6}')
        else:
            print(f'{mA} vs {mB:<20}  (predictions not stored in results.json)')